# Phase 6 Analysis — Paper Trading Results

**Период**: 2026-03-08 → 2026-03-16 (8 дней из запланированных 14)

**Цель**: Оценить HTR модель в live-условиях, сравнить с backtest, определить go/no-go для Phase 7.

**Инстансы**:
- `main` — базовый HTR, no crypto/commodity, 14 исходных токенов + auto-refresh
- `v1_baseline` — auto-discover, no crypto/commodity, чистый старт
- `v2` — 5 улучшений (penny filter, edge cooldown, adverse 10%, category time exit, trailing)
- `small_markets` — маленькие рынки (liq < $100K, ascending volume)
- `small_longshot` — лонгшоты на малых рынках (p=0.10–0.45)
- `sports_only` — только спорт (RETIRED, WR=6%)
- `inverse` — инвертированный сигнал (RETIRED, sanity check)

**Источники знаний**:
- Ernest Chan "Quantitative Trading" Ch.6-7 (Kelly, stop loss, MR)
- López de Prado "ML for Asset Managers" Ch.5-8 (meta-labeling, Deflated SR)
- Phase 5.5 experiments (CFI, Focal Loss, meta-labeling, NeuralForecast)
- Phase 2 EDA: 77% markets are MR (VR<1), ADF 19% stationary

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
SEED = 42
np.random.seed(SEED)

# Paths — данные уже скопированы с VPS
LOGS = Path('../../logs/paper_trading')
START_CAPITAL = 1000.0

print('Phase 6 Analysis loaded')

## 1. Загрузка данных со всех инстансов

In [ ]:
def load_checkpoint(path):
    """Load checkpoint.json → trades, positions, cash."""
    with open(path) as f:
        d = json.load(f)
    trades = pd.DataFrame(d.get('trades', []))
    positions = d.get('positions', {})
    cash = d.get('cash', START_CAPITAL)
    return trades, positions, cash

def load_equity(directory):
    """Load latest equity CSV from directory."""
    csvs = sorted(directory.glob('paper_equity_*.csv'))
    if not csvs:
        return None
    df = pd.read_csv(csvs[-1])
    df['time'] = pd.to_datetime(df['time'])
    return df

# Load all instances
instances = {}
configs = {
    'main': {'cp': LOGS / 'checkpoint.json', 'eq_dir': LOGS},
    'v1_baseline': {'cp': LOGS / 'v1_baseline' / 'checkpoint.json', 'eq_dir': LOGS / 'v1_baseline'},
    'v2': {'cp': LOGS / 'v2' / 'checkpoint.json', 'eq_dir': LOGS / 'v2'},
    'small_markets': {'cp': LOGS / 'small_markets' / 'checkpoint.json', 'eq_dir': LOGS / 'small_markets'},
    'small_longshot': {'cp': LOGS / 'small_longshot' / 'checkpoint.json', 'eq_dir': LOGS / 'small_longshot'},
}

# Archived
archive = LOGS / 'archive'
if archive.exists():
    for d in sorted(archive.iterdir()):
        cp = d / 'checkpoint.json'
        if cp.exists():
            name = d.name.rsplit('_', 2)[0]  # sports_only_20260314_... → sports_only
            configs[name] = {'cp': cp, 'eq_dir': d}

for name, cfg in configs.items():
    try:
        trades, positions, cash = load_checkpoint(cfg['cp'])
        equity = load_equity(cfg['eq_dir'])
        instances[name] = {
            'trades': trades, 'positions': positions,
            'cash': cash, 'equity': equity,
        }
        n_trades = len(trades)
        pnl = trades['pnl'].sum() if n_trades > 0 else 0
        wr = (trades['pnl'] > 0).mean() * 100 if n_trades > 0 else 0
        print(f'{name:20s}: {n_trades:3d} trades, PnL ${pnl:+.2f}, WR {wr:.0f}%, '
              f'{len(positions)} open, cash ${cash:.0f}')
    except Exception as e:
        print(f'{name:20s}: FAILED — {e}')

# Primary analysis on main (most data)
main_trades = instances['main']['trades'].copy()
main_equity = instances['main']['equity'].copy()
print(f'\nMain: {len(main_trades)} trades, equity CSV {len(main_equity)} rows')

## 2. Summary Table — все инстансы

In [ ]:
rows = []
for name, data in instances.items():
    t = data['trades']
    if len(t) == 0:
        continue
    pnl = t['pnl'].sum()
    wins = (t['pnl'] > 0).sum()
    losses = (t['pnl'] <= 0).sum()
    wr = wins / len(t) * 100
    avg_win = t.loc[t['pnl'] > 0, 'pnl'].mean() if wins > 0 else 0
    avg_loss = t.loc[t['pnl'] <= 0, 'pnl'].mean() if losses > 0 else 0
    pf = abs(t.loc[t['pnl'] > 0, 'pnl'].sum() / t.loc[t['pnl'] <= 0, 'pnl'].sum()) if losses > 0 else np.inf
    # Max drawdown from equity
    eq = data.get('equity')
    if eq is not None and len(eq) > 0:
        eqv = eq['mtm_equity'].values
        peak = np.maximum.accumulate(eqv)
        dd = np.where(peak > 0, (eqv - peak) / peak * 100, 0)
        max_dd = dd.min()
    else:
        max_dd = 0
    rows.append({
        'Instance': name, 'Trades': len(t), 'Wins': wins, 'Losses': losses,
        'WR %': f'{wr:.1f}', 'PnL $': f'{pnl:+.2f}',
        'Avg Win $': f'{avg_win:.2f}', 'Avg Loss $': f'{avg_loss:.2f}',
        'PF': f'{pf:.2f}' if pf < 100 else 'inf',
        'Max DD %': f'{max_dd:.1f}',
        'Open': len(data['positions']),
    })

summary = pd.DataFrame(rows).set_index('Instance')
summary

## 3. Equity Curves

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = {'main': '#1f77b4', 'v1_baseline': '#aec7e8', 'v2': '#ff7f0e',
          'small_markets': '#9467bd', 'small_longshot': '#8c564b',
          'sports_only': '#2ca02c', 'inverse': '#d62728'}

for name, data in instances.items():
    eq = data.get('equity')
    if eq is not None and len(eq) > 10:
        ax.plot(eq['time'], eq['mtm_equity'], label=name,
                color=colors.get(name, 'gray'),
                linewidth=2.5 if name == 'main' else 1.2)

ax.axhline(y=START_CAPITAL, color='gray', linestyle='--', alpha=0.5, label='Start ($1000)')
ax.set_title('Phase 6: Equity Curves — All Instances')
ax.set_ylabel('Equity ($)')
ax.legend(loc='lower left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Main Instance — Deep Analysis

Main имеет больше всего данных (248 trades, 8 дней). Это наш primary dataset.

In [ ]:
# Parse timestamps
for col in ['entry_time', 'exit_time']:
    if col in main_trades.columns:
        main_trades[col] = pd.to_datetime(main_trades[col])

# Hold time
if 'entry_time' in main_trades.columns and 'exit_time' in main_trades.columns:
    main_trades['hold_hours'] = (main_trades['exit_time'] - main_trades['entry_time']).dt.total_seconds() / 3600

# Daily PnL
if 'exit_time' in main_trades.columns:
    main_trades['exit_date'] = main_trades['exit_time'].dt.date

# PnL distribution
pnls = main_trades['pnl'].values
print(f'=== PnL Distribution (Main, N={len(pnls)}) ===')
print(f'Mean:   ${np.mean(pnls):+.2f}')
print(f'Median: ${np.median(pnls):+.2f}')
print(f'Stdev:  ${np.std(pnls):.2f}')
print(f'Skew:   {pd.Series(pnls).skew():.3f}')
print(f'Kurt:   {pd.Series(pnls).kurtosis():.3f}')
print(f'Min:    ${pnls.min():+.2f}')
print(f'Max:    ${pnls.max():+.2f}')
print(f'\nPercentiles:')
for q in [5, 25, 50, 75, 95]:
    print(f'  P{q}: ${np.percentile(pnls, q):+.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. PnL histogram
axes[0].hist(pnls, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', linestyle='--')
axes[0].axvline(x=np.mean(pnls), color='green', linestyle='--', label=f'Mean ${np.mean(pnls):+.2f}')
axes[0].set_title('PnL Distribution')
axes[0].set_xlabel('PnL ($)')
axes[0].legend()

# 2. Cumulative PnL
cum_pnl = np.cumsum(pnls)
axes[1].plot(range(1, len(cum_pnl)+1), cum_pnl, color='steelblue')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].fill_between(range(1, len(cum_pnl)+1), cum_pnl, 0,
                     where=cum_pnl >= 0, color='green', alpha=0.2)
axes[1].fill_between(range(1, len(cum_pnl)+1), cum_pnl, 0,
                     where=cum_pnl < 0, color='red', alpha=0.2)
axes[1].set_title('Cumulative PnL by Trade #')
axes[1].set_xlabel('Trade #')
axes[1].set_ylabel('$')

# 3. Daily PnL
if 'exit_date' in main_trades.columns:
    daily = main_trades.groupby('exit_date')['pnl'].agg(['sum', 'count'])
    colors_daily = ['green' if x >= 0 else 'red' for x in daily['sum']]
    axes[2].bar(range(len(daily)), daily['sum'], color=colors_daily, alpha=0.8)
    axes[2].set_xticks(range(len(daily)))
    axes[2].set_xticklabels([str(d)[-5:] for d in daily.index], rotation=45)
    axes[2].axhline(y=0, color='gray', linestyle='--')
    axes[2].set_title('Daily PnL')
    axes[2].set_ylabel('$')
    for i, (s, c) in enumerate(zip(daily['sum'], daily['count'])):
        axes[2].annotate(f'{int(c)}t', (i, s), ha='center',
                        va='bottom' if s >= 0 else 'top', fontsize=8)

plt.tight_layout()
plt.show()

## 5. PnL by Exit Reason — КЛЮЧЕВОЙ анализ

Из backtest (Phase 5): HTR = hold-to-resolution, $8.61/trade flat-bet edge.  
В paper trading добавлены промежуточные exit'ы. Вопрос: **помогают они или вредят?**

In [ ]:
reason_col = 'exit_reason' if 'exit_reason' in main_trades.columns else 'reason'
if reason_col in main_trades.columns:
    by_reason = main_trades.groupby(reason_col)['pnl'].agg(['sum', 'count', 'mean'])
    by_reason = by_reason.sort_values('sum')
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Total PnL by reason
    colors_r = ['green' if x >= 0 else 'red' for x in by_reason['sum']]
    bars = axes[0].barh(range(len(by_reason)), by_reason['sum'], color=colors_r, alpha=0.8)
    axes[0].set_yticks(range(len(by_reason)))
    axes[0].set_yticklabels(by_reason.index, fontsize=9)
    axes[0].axvline(x=0, color='gray', linestyle='--')
    axes[0].set_title('Total PnL by Exit Reason')
    axes[0].set_xlabel('$')
    for i, (s, c) in enumerate(zip(by_reason['sum'], by_reason['count'])):
        axes[0].text(s + (5 if s >= 0 else -5), i, f'${s:+.0f} ({c})',
                    va='center', ha='left' if s >= 0 else 'right', fontsize=8)
    
    # Avg PnL by reason
    colors_a = ['green' if x >= 0 else 'red' for x in by_reason['mean']]
    axes[1].barh(range(len(by_reason)), by_reason['mean'], color=colors_a, alpha=0.8)
    axes[1].set_yticks(range(len(by_reason)))
    axes[1].set_yticklabels(by_reason.index, fontsize=9)
    axes[1].axvline(x=0, color='gray', linestyle='--')
    axes[1].set_title('Avg PnL per Trade by Exit Reason')
    axes[1].set_xlabel('$/trade')
    
    plt.tight_layout()
    plt.show()
    
    print('\nДетали:')
    for idx, row in by_reason.iterrows():
        wr = (main_trades[main_trades[reason_col] == idx]['pnl'] > 0).mean() * 100
        print(f'  {idx:30s}: {int(row["count"]):3d} trades, '
              f'${row["sum"]:+8.2f} total, ${row["mean"]:+6.2f}/trade, WR {wr:.0f}%')

### 5.1 Вывод по exit reasons

**MR target hit** — единственный прибыльный exit (+$480, 64 trades).  
Все остальные exit'ы **уничтожают edge**:

| Exit | PnL | N | Проблема |
|------|-----|---|----------|
| Resolution loss | -$128 | 8 | Модель предсказала неправильную сторону → max loss |
| Adverse move | -$126 | 11 | Выходит при откате 10%, но MR рынки ДОЛЖНЫ откатываться |
| Edge gone | -$122 | 32 | Выходит когда edge<0.5% → шум, edge вернётся |
| Time exit | -$77 | 65 | 12h слишком коротко для MR (median hold = 158h в backtest!) |
| Trailing exit | -$64 | 59 | Фиксирует прибыль слишком рано |

**Корень проблемы**: backtest использовал чистый HTR (hold-to-resolution, median 158h).  
Paper trading добавил exit'ы, которые **cutting winners short** — нарушение главного правила MR торговли.

> **Chan Ch.6**: "Stop loss вреден для mean-reverting стратегий".  
> **Наш Phase 2**: 77% markets are MR (VR<1), 93% have VR<1.  
> Все промежуточные exit'ы = stop loss в другой форме → ВРЕДНЫ.

## 6. Backtest vs Paper Trading — расхождение

In [ ]:
# Backtest metrics (from Phase 5: notebooks 11-12)
backtest = {
    'Period': 'Phase 5 (backtest)',
    'Trades': 688,
    'WR %': 73,
    'Avg PnL/trade $': 8.61,
    'Total PnL $': 5924,
    'Profit Factor': 2.22,
    'Flat-bet p-value': 0.003,
    'Avg Hold (hours)': 158,
    'Exit': 'Resolution only',
    'Model': 'HTR v1 (AUC=0.959)',
}

paper = {
    'Period': 'Phase 6 (paper)',
    'Trades': len(main_trades),
    'WR %': round((main_trades['pnl'] > 0).mean() * 100, 1),
    'Avg PnL/trade $': round(main_trades['pnl'].mean(), 2),
    'Total PnL $': round(main_trades['pnl'].sum(), 2),
    'Profit Factor': round(
        abs(main_trades.loc[main_trades['pnl'] > 0, 'pnl'].sum() /
            main_trades.loc[main_trades['pnl'] <= 0, 'pnl'].sum()), 2
    ) if (main_trades['pnl'] <= 0).any() else 'inf',
    'Avg Hold (hours)': round(main_trades['hold_hours'].mean(), 1) if 'hold_hours' in main_trades.columns else '?',
    'Exit': 'Multiple (MR/trailing/time/edge/adverse)',
    'Model': 'HTR v1 (same)',
}

comparison = pd.DataFrame([backtest, paper]).set_index('Period').T
comparison

### 6.1 Root Causes расхождения

**1. Exit mechanism mismatch (ГЛАВНАЯ ПРИЧИНА)**

Backtest: hold-to-resolution (median 158h). Единственный exit = resolution.  
Paper: 6 разных exit'ов, median hold 2.2h (!). Разница в 72x.

MR target (+$480) подтверждает что HTR модель РАБОТАЕТ — но только если дать позиции время.  
Остальные exit'ы закрывают позиции **до того как MR завершится**.

**2. Time horizon mismatch**

HTR модель обучена предсказывать resolution outcome (дни/недели).  
Paper trader оценивает edge каждые 60 секунд и выходит при малейшем изменении.  
Это как использовать годовой прогноз погоды для решения — брать ли зонт через 5 минут.

**3. Re-entry noise**

93% трейдов main — re-entries в те же рынки (Iran 15x, etc.).
Cooldown 30min → exit edge gone → re-enter 30min later → exit again.  
Каждый цикл стоит ~2× fee (entry + exit) = 3.5% от позиции.

**4. Backtest optimism (López de Prado Ch.8: Deflated SR)**

Backtest SR был получен ПОСЛЕ множественного тестирования (Optuna 50 trials × 2 models).  
По Deflated SR формуле, при N_trials=100, реальный SR deflates значительно.

## 7. Deflated Sharpe Ratio (López de Prado Ch.8)

In [ ]:
from scipy import stats

def deflated_sr(sr_observed, n_trials, n_obs, skew=0, kurtosis=3):
    """López de Prado Deflated Sharpe Ratio.
    
    Tests H0: SR* ≤ E[max(SR)] where SR* is observed and 
    E[max] is expected max SR from N random trials.
    """
    # Expected max SR from N random trials (Euler-Mascheroni)
    euler_mascheroni = 0.5772
    e_max_sr = np.sqrt(2 * np.log(n_trials)) - \
               (np.log(np.pi) + euler_mascheroni) / (2 * np.sqrt(2 * np.log(n_trials)))
    
    # SR standard error (Bailey & López de Prado, 2012)
    se_sr = np.sqrt((1 - skew * sr_observed + (kurtosis - 1) / 4 * sr_observed**2) / n_obs)
    
    # Test statistic
    z = (sr_observed - e_max_sr) / se_sr
    p_value = stats.norm.cdf(z)
    
    return {
        'SR_observed': sr_observed,
        'E_max_SR': e_max_sr,
        'SE_SR': se_sr,
        'z': z,
        'p_value': p_value,
        'significant': p_value > 0.95,  # reject H0 at 5%
    }

# Paper trading SR
paper_returns = main_trades['pnl'].values / START_CAPITAL  # normalize
paper_sr = np.mean(paper_returns) / np.std(paper_returns) * np.sqrt(252 * 24)  # annualized (hourly)

# Number of trials (configs tested during development)
n_trials = 100  # conservative: Optuna 50 + manual 50

result = deflated_sr(
    sr_observed=paper_sr,
    n_trials=n_trials,
    n_obs=len(paper_returns),
    skew=pd.Series(paper_returns).skew(),
    kurtosis=pd.Series(paper_returns).kurtosis() + 3,
)

print('=== Deflated Sharpe Ratio (López de Prado Ch.8) ===')
for k, v in result.items():
    if isinstance(v, float):
        print(f'  {k:15s}: {v:.4f}')
    else:
        print(f'  {k:15s}: {v}')

print(f'\nВывод: SR observed = {result["SR_observed"]:.3f}')
print(f'E[max SR from {n_trials} random trials] = {result["E_max_SR"]:.3f}')
if not result['significant']:
    print('→ Наблюдаемый SR НЕ превышает ожидаемый от случайных стратегий')
    print('→ Нельзя отвергнуть H0: модель не лучше random')
else:
    print('→ SR значимо выше случайного')

## 8. По категориям рынков

In [ ]:
# Classify trades by category
def classify_market(question):
    if not isinstance(question, str):
        return 'unknown'
    q = question.lower()
    sports_kw = ['vs.', 'spread:', 'o/u', 'nba', 'nfl', 'nhl', 'mlb', 'lol:',
                 'counter-strike', 'dota', 'esport', 'league of legends',
                 'playoffs', 'premier league', 'la liga', 'serie a',
                 'champions league', 'fc ', 'win the 202', 'world cup',
                 'stanley cup', 'super bowl']
    geo_kw = ['iran', 'russia', 'ukraine', 'war', 'ceasefire', 'regime',
              'forces', 'strike', 'military', 'invasion', 'offensive', 'lebanon']
    politics_kw = ['president', 'trump', 'democrat', 'republican', 'congress',
                   'governor', 'nomination', 'rubio', 'newsom', 'election']
    crypto_kw = ['bitcoin', 'btc', 'ethereum', 'eth', 'crypto']
    commodity_kw = ['crude oil', 'wti', 'brent', 'natural gas', 'gold price']
    
    for kw in crypto_kw:
        if kw in q: return 'crypto'
    for kw in commodity_kw:
        if kw in q: return 'commodity'
    for kw in sports_kw:
        if kw in q: return 'sports'
    for kw in geo_kw:
        if kw in q: return 'geopolitics'
    for kw in politics_kw:
        if kw in q: return 'politics'
    return 'other'

q_col = 'market_question' if 'market_question' in main_trades.columns else 'question'
if q_col in main_trades.columns:
    main_trades['category'] = main_trades[q_col].apply(classify_market)
    
    cat_stats = main_trades.groupby('category').agg(
        trades=('pnl', 'count'),
        total_pnl=('pnl', 'sum'),
        avg_pnl=('pnl', 'mean'),
        wr=('pnl', lambda x: (x > 0).mean() * 100),
    ).sort_values('total_pnl')
    
    fig, ax = plt.subplots(figsize=(10, 4))
    colors_cat = ['green' if x >= 0 else 'red' for x in cat_stats['total_pnl']]
    ax.barh(cat_stats.index, cat_stats['total_pnl'], color=colors_cat, alpha=0.8)
    ax.axvline(x=0, color='gray', linestyle='--')
    ax.set_title('PnL by Market Category (Main Instance)')
    ax.set_xlabel('Total PnL ($)')
    for i, (idx, row) in enumerate(cat_stats.iterrows()):
        ax.text(row['total_pnl'] + (3 if row['total_pnl'] >= 0 else -3), i,
                f'${row["total_pnl"]:+.0f} ({int(row["trades"])}t, WR {row["wr"]:.0f}%)',
                va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
    
    print(cat_stats.round(1))

## 9. Hold Time Analysis

Backtest: median hold = 158h. Paper trading: median = 2.2h.  
Проверим: зависит ли PnL от hold time?

In [ ]:
if 'hold_hours' in main_trades.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Hold time distribution
    axes[0].hist(main_trades['hold_hours'], bins=30, color='steelblue', edgecolor='white')
    axes[0].axvline(x=main_trades['hold_hours'].median(), color='red', linestyle='--',
                   label=f'Median {main_trades["hold_hours"].median():.1f}h')
    axes[0].set_title('Hold Time Distribution')
    axes[0].set_xlabel('Hours')
    axes[0].legend()
    
    # PnL vs hold time
    wins = main_trades[main_trades['pnl'] > 0]
    losses = main_trades[main_trades['pnl'] <= 0]
    axes[1].scatter(losses['hold_hours'], losses['pnl'], c='red', alpha=0.4, s=20, label='Loss')
    axes[1].scatter(wins['hold_hours'], wins['pnl'], c='green', alpha=0.4, s=20, label='Win')
    axes[1].axhline(y=0, color='gray', linestyle='--')
    axes[1].set_title('PnL vs Hold Time')
    axes[1].set_xlabel('Hours')
    axes[1].set_ylabel('PnL ($)')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # PnL by hold time bucket
    bins = [0, 1, 3, 6, 12, 24, 999]
    labels = ['<1h', '1-3h', '3-6h', '6-12h', '12-24h', '>24h']
    main_trades['hold_bucket'] = pd.cut(main_trades['hold_hours'], bins=bins, labels=labels)
    bucket_stats = main_trades.groupby('hold_bucket', observed=False).agg(
        n=('pnl', 'count'),
        total_pnl=('pnl', 'sum'),
        avg_pnl=('pnl', 'mean'),
        wr=('pnl', lambda x: (x > 0).mean() * 100 if len(x) > 0 else 0),
    )
    print('PnL by hold time bucket:')
    print(bucket_stats.round(2))

## 10. A/B Test Results — сравнение стратегий

In [ ]:
# Collect all instance results for comparison
ab_results = []
for name, data in instances.items():
    t = data['trades']
    if len(t) == 0:
        continue
    pnl = t['pnl'].sum()
    ab_results.append({
        'Instance': name,
        'Strategy': {
            'main': 'HTR + all exits',
            'v1_baseline': 'HTR baseline (fresh start)',
            'v2': 'HTR + 5 fixes (penny, cooldown, adverse, time, trailing)',
            'small_markets': 'HTR on small markets (liq<$100K)',
            'small_longshot': 'HTR longshots (p<0.45, liq<$100K)',
            'sports_only': 'HTR sports only',
            'inverse': 'Inverted HTR (sanity)',
        }.get(name, '?'),
        'Trades': len(t),
        'PnL': pnl,
        'PnL/trade': pnl / len(t),
        'WR': (t['pnl'] > 0).mean() * 100,
        'Status': 'LOSS' if pnl < 0 else 'PROFIT',
    })

ab_df = pd.DataFrame(ab_results).sort_values('PnL', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors_ab = ['green' if x >= 0 else 'red' for x in ab_df['PnL']]
bars = ax.barh(ab_df['Instance'], ab_df['PnL'], color=colors_ab, alpha=0.8)
ax.axvline(x=0, color='gray', linestyle='--')
ax.set_title('Phase 6 A/B Test: PnL by Instance')
ax.set_xlabel('Total PnL ($)')
for i, (_, row) in enumerate(ab_df.iterrows()):
    ax.text(row['PnL'] + (2 if row['PnL'] >= 0 else -2), i,
            f'${row["PnL"]:+.0f} ({int(row["Trades"])}t, WR={row["WR"]:.0f}%)',
            va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\nA/B Results:')
print(ab_df[['Instance', 'Strategy', 'Trades', 'PnL', 'PnL/trade', 'WR', 'Status']].to_string(index=False))

## 11. Что работает, а что нет — итоги A/B

### Работает:
- **MR target exit** — единственный прибыльный механизм (+$480 на 64 трейда)
- **Геополитика (некоторые рынки)** — Iran ceasefire +$21.85 (14 трейдов)
- **Спортивные спреды** — некоторые profitable

### Не работает:
- **Все промежуточные exit'ы** — суммарно -$500+ (edge gone, time, trailing, adverse)
- **Спорт как категория** — WR=6% (sports_only), модель не имеет edge
- **Маленькие рынки** — WR=7-23%, менее эффективны но модель не эксплуатирует
- **Longshots** — WR=7%, массивные потери
- **Re-entry loop** — 93% трейдов main = повторные входы в одни рынки, каждый стоит 2× fee
- **v2 fixes** — 5 улучшений не помогли (WR=30% vs 47% baseline)

## 12. Simulated HTR-only (What If?)

Если бы мы держали каждую позицию до resolution (как в backtest),  
а не выходили по time/edge/trailing — каков был бы результат?

Оценка: MR target = proxy для resolution win.  
64 MR targets из 248 → 26% conversion rate.  
Остальные 184 трейда были закрыты промежуточными exit'ами.  
Если бы они дошли до resolution с тем же WR (47%), результат был бы:

In [ ]:
# Simulation: what if we held all positions to MR target or resolution?
mr_trades = main_trades[main_trades[reason_col].str.contains('MR target', na=False)] if reason_col in main_trades.columns else pd.DataFrame()
non_mr = main_trades[~main_trades[reason_col].str.contains('MR target', na=False)] if reason_col in main_trades.columns else pd.DataFrame()

if len(mr_trades) > 0:
    print(f'MR target trades: {len(mr_trades)}')
    print(f'  Total PnL: ${mr_trades["pnl"].sum():+.2f}')
    print(f'  Avg PnL: ${mr_trades["pnl"].mean():+.2f}')
    print(f'  WR: {(mr_trades["pnl"] > 0).mean()*100:.0f}%')
    print(f'  Avg hold: {mr_trades["hold_hours"].mean():.1f}h')
    print()
    print(f'Non-MR trades (closed early): {len(non_mr)}')
    print(f'  Total PnL: ${non_mr["pnl"].sum():+.2f}')
    print(f'  Avg PnL: ${non_mr["pnl"].mean():+.2f}')
    print(f'  WR: {(non_mr["pnl"] > 0).mean()*100:.0f}%')
    print(f'  Avg hold: {non_mr["hold_hours"].mean():.1f}h')
    print()
    print('=== Counterfactual: если бы ВСЕ trades = MR target ===')
    print(f'  Expected PnL: 248 × ${mr_trades["pnl"].mean():+.2f} = ${248 * mr_trades["pnl"].mean():+.0f}')
    print(f'  vs actual: ${main_trades["pnl"].sum():+.2f}')
    print(f'  Lost edge from early exits: ${248 * mr_trades["pnl"].mean() - main_trades["pnl"].sum():+.0f}')

## 13. Recommendations — что делать дальше

### Вывод Phase 6: **NO GO для Phase 7 (live trading)**

Все 7 конфигураций убыточны. HTR модель в текущей реализации paper trader не генерирует edge.

### Причины и решения:

**Проблема 1: Exit mechanisms уничтожают edge**
- MR target = единственный profitable exit (+$480)
- Все остальные (edge gone, time, trailing, adverse) = -$510 суммарно
- **Решение**: Для HTR стратегии — убрать ВСЕ промежуточные exit'ы. HTR = Hold To Resolution.
  Оставить только: resolution + daily portfolio stop (safety net)

**Проблема 2: Time horizon mismatch**  
- HTR обучен на resolution outcomes (дни/недели)
- Paper trader evaluates каждые 60 сек с hold median 2.2h
- **Решение**: Если HTR → hold до resolution. Если хотим short-term → нужна ДРУГАЯ модель
  (MR z-score strategy из Phase 5, которая имеет свой exit mechanism)

**Проблема 3: Re-entry cost**
- 93% трейдов = re-entries, каждый стоит 2× fee = 3.5%
- **Решение**: Cooldown = infinity (never re-enter same token). HTR = one shot per market.

**Проблема 4: No meta-labeling**
- Phase 5.5 показал: meta-labeling WR 60%→78% при P≥0.6, Sharpe 0.21→0.50
- Paper trading не использовал meta-labeling
- **Решение**: Включить meta-filter в Phase 6 v2

**Проблема 5: Категории без edge**
- Sports: WR=6% (модель не предсказывает)
- Small markets / longshots: WR=7-23%
- **Решение**: Фильтр категорий на основе Phase 6 data (только geopolitics+politics)

### Plan — Phase 6 v2 (итерация):

1. **Pure HTR mode**: убрать ВСЕ exits кроме resolution + portfolio stop
2. **Meta-labeling filter**: P(correct) ≥ 0.6 → trade, else skip
3. **No re-entry**: cooldown = ∞ для закрытых токенов
4. **Category filter**: exclude sports, crypto, commodity; focus geopolitics+politics
5. **Longer test**: 14 дней minimum
6. **Deflated SR validation**: если SR > E[max] → proceed to Phase 7

In [ ]:
print('=' * 70)
print('PHASE 6 VERDICT: NO GO FOR PHASE 7')
print('=' * 70)
print()
print('Key findings:')
print('  1. ALL 7 configurations are UNPROFITABLE')
print(f'  2. Main: {len(main_trades)} trades, PnL ${main_trades["pnl"].sum():+.2f}, WR {(main_trades["pnl"]>0).mean()*100:.0f}%')
print(f'  3. MR target is ONLY profitable exit (+$480, 64 trades)')
print(f'  4. Early exits DESTROY edge (-$510 across 184 trades)')
print(f'  5. Backtest vs paper: $8.61/trade → $-0.10/trade')
print(f'  6. Sports WR=6%, Longshots WR=7% — no edge')
print()
print('Root cause: EXIT MECHANISM MISMATCH')
print('  Backtest: hold-to-resolution (median 158h)')
print(f'  Paper: median hold {main_trades["hold_hours"].median():.1f}h — 72x shorter')
print()
print('Next: Phase 6 v2 — pure HTR (no early exits) + meta-labeling filter')

---

## Appendix: Связь с исследованиями

### Ernest Chan "Quantitative Trading" (Ch.6-7)
- **Подтверждено**: stop loss вреден для MR (наши exit'ы = формы stop loss → убыточны)
- **Подтверждено**: half-Kelly sizing работает (position sizes адекватны)
- **Подтверждено**: simple > complex (MR target = простейший exit, единственный profitable)

### López de Prado "ML for Asset Managers" (Ch.5-8)
- **Meta-labeling** (Ch.5): НЕ использовали в paper → potential +50% edge improvement
- **Deflated SR** (Ch.8): paper trading SR не значим → нельзя доверять backtest SR
- **CFI** (Ch.6): 10 noise features удалены, но модель всё равно не даёт edge в live

### Phase 5.5 Experiments
- **Focal Loss**: recall UP 19%→89%, но в paper trading class balance другой
- **Meta-labeling**: WR 60%→78% — ГЛАВНОЕ неиспользованное улучшение
- **Trend-scanning**: negative result подтверждён (MR markets)
- **NeuralForecast**: convergence bias, LGB remains best

### Phase 2 EDA
- **77% markets are MR** → exit mechanisms ВРЕДНЫ (подтверждено Phase 6)
- **YES bias +0.217** → contrarian strategy потенциально лучше HTR
- **Whale Gini 0.918** → рынки контролируются крупными игроками → трудно предсказать